In [ ]:
# ============================================================
# 1. Environment setup  --  MUST run before anything imports paddle
# ============================================================
import importlib.util
import os
import subprocess
import sys

# CPU-only flags: disable PIR + oneDNN/MKLDNN to avoid the known CPU
# inference issue. Harmless (ignored) when running on GPU.
os.environ["FLAGS_enable_pir_api"] = "0"
os.environ["FLAGS_use_mkldnn"] = "0"
os.environ["PADDLE_PDX_ENABLE_MKLDNN_BYDEFAULT"] = "0"

# Install only what is missing -- a no-op on a machine that already has them.
for pkg, mod in [("pymupdf", "fitz"), ("opencv-python", "cv2"),
                 ("pandas", "pandas"), ("paddleocr", "paddleocr")]:
    if importlib.util.find_spec(mod) is None:
        print(f"installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                       check=True)

# ---------------------------------------------------------------------------
# Import order matters when torch is also installed (typical local machine).
# paddle and torch ship conflicting bundled DLLs, so on Windows:
#   import paddle -> import torch   ==  OSError [WinError 127] on shm.dll
#   import torch  -> import paddle  ==  fine
# paddlex runs a langchain shim on import that reaches torch via transformers,
# so importing paddleocr first pins the working order for the whole session.
# Harmless in Colab, required locally.
# ---------------------------------------------------------------------------
if "paddle" in sys.modules and "torch" not in sys.modules:
    raise RuntimeError(
        "paddle was imported before torch in this kernel -- restart the kernel "
        "and run this cell first."
    )

import paddleocr  # noqa: F401  -- MUST come before `import paddle`

print("Environment ready.")

In [ ]:
# ============================================================
# 2. Sanity check
# ============================================================
import paddle

print("paddle:", paddle.__version__)
print("device:", paddle.get_device())

In [ ]:
# ============================================================
# 3. Settings  --  the only cell you normally need to touch
# ============================================================

PDF_DPI    = 200      # rendering resolution of each PDF page
START_PAGE = 1        # 1-based index of the first page to process
NUM_PAGES  = 5        # how many pages to run layout detection on

# Titles on these scanned pages score between 0.25 and 0.40, so anything
# above ~0.4 silently drops every paragraph_title / figure_title / image.
# Keep this low and let LAYOUT_NMS clean up the duplicates instead.
THRESHOLD  = 0.25

# Per-class override -- set to None to use the single THRESHOLD above.
# Observed class ids: 0=paragraph_title, 1=image, 2=text, 3=number, 6=figure_title
THRESHOLD_PER_CLASS = None
# THRESHOLD_PER_CLASS = {0: 0.20, 1: 0.30, 2: 0.30, 3: 0.30, 6: 0.20}

LAYOUT_NMS = True     # drops the duplicate/overlapping boxes of the same class

# Binarize each page before detection. The old good results came from
# pre-binarized images (page_XXXX_05_binary.png); raw PDF renders have a
# grey background that lowers every confidence score.
BINARIZE = True

PAGES_DIR  = "./pdf_pages"      # rendered page images go here
OUTPUT_DIR = "./layout_output"  # layout results (images + json + stats) go here

_thr = THRESHOLD_PER_CLASS if THRESHOLD_PER_CLASS is not None else THRESHOLD
print(f"Pages {START_PAGE}..{START_PAGE + NUM_PAGES - 1} | dpi={PDF_DPI} | "
      f"threshold={_thr} | nms={LAYOUT_NMS} | binarize={BINARIZE}")

In [ ]:
# ============================================================
# 4. Get the PDF  --  works both in Colab and locally
# ============================================================
import glob
import os

# Local run: set this to your PDF path. Leave as None to auto-pick the first
# *.pdf sitting next to the notebook (in Colab it opens the upload widget).
PDF_PATH = None

try:
    from google.colab import files as colab_files  # aliased, keeps `files` free
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if PDF_PATH:
    pdf_path = PDF_PATH
elif IN_COLAB:
    uploaded = colab_files.upload()
    pdf_path = next(iter(uploaded.keys()))
else:
    candidates = sorted(glob.glob("*.pdf"))
    if not candidates:
        raise FileNotFoundError(
            "No PDF found. Put one next to the notebook, or set PDF_PATH above."
        )
    pdf_path = candidates[0]
    if len(candidates) > 1:
        print(f"Found {len(candidates)} PDFs, using the first one. "
              f"Set PDF_PATH to choose: {candidates}")

if not os.path.exists(pdf_path):
    raise FileNotFoundError(pdf_path)

print(f"Environment: {'Colab' if IN_COLAB else 'local'}")
print(f"PDF: {pdf_path}")

In [ ]:
# ============================================================
# 5. Split the PDF into page images  (+ optional binarization)
# ============================================================
import os
import cv2
import numpy as np
import fitz  # PyMuPDF

os.makedirs(PAGES_DIR, exist_ok=True)

doc = fitz.open(pdf_path)
total_pages = doc.page_count
print(f"PDF has {total_pages} page(s). Rendering at {PDF_DPI} dpi ...")

matrix = fitz.Matrix(PDF_DPI / 72, PDF_DPI / 72)  # 72 dpi is the PDF base unit

page_paths = []
for i in range(total_pages):
    pix = doc[i].get_pixmap(matrix=matrix)
    out_path = os.path.join(PAGES_DIR, f"page_{i + 1:04d}.png")

    if BINARIZE:
        # pixmap -> numpy -> grey -> Otsu threshold -> back to 3-channel PNG
        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(
            pix.height, pix.width, pix.n
        )
        grey = cv2.cvtColor(img[:, :, :3], cv2.COLOR_RGB2GRAY)
        _, binary = cv2.threshold(
            grey, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )
        cv2.imwrite(out_path, cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR))
    else:
        pix.save(out_path)

    page_paths.append(out_path)

doc.close()

print(f"Saved {len(page_paths)} page image(s) -> {PAGES_DIR} "
      f"(binarize={BINARIZE})")
print("First few:", page_paths[:3])

In [ ]:
# ============================================================
# 6. Load the layout model  (once)
# ============================================================
from paddleocr import LayoutDetection

model = LayoutDetection(
    model_name="PP-DocLayout_plus-L",
    threshold=THRESHOLD_PER_CLASS if THRESHOLD_PER_CLASS is not None else THRESHOLD,
    layout_nms=LAYOUT_NMS,
)

print("Layout model loaded successfully!")

In [ ]:
# ============================================================
# 7. Run layout detection on the selected pages
# ============================================================
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

start_idx = max(START_PAGE - 1, 0)
selected = page_paths[start_idx : start_idx + NUM_PAGES]

if not selected:
    raise ValueError(
        f"No pages selected. PDF has {len(page_paths)} page(s), "
        f"but START_PAGE={START_PAGE}."
    )

print(f"Running layout detection on {len(selected)} page(s) ...\n")

records = []  # one row per detected box

for path in selected:
    page_name = os.path.splitext(os.path.basename(path))[0]
    page_dir = os.path.join(OUTPUT_DIR, page_name)
    os.makedirs(page_dir, exist_ok=True)

    n_boxes = 0
    for res in model.predict(path, batch_size=1):
        res.save_to_img(save_path=page_dir)
        res.save_to_json(save_path=os.path.join(page_dir, "result.json"))

        # NOTE: DetResult is dict-like with the keys at the top level.
        # There is no "res" wrapper key -- that only appears in res.print().
        for box in res["boxes"]:
            x1, y1, x2, y2 = (float(v) for v in box["coordinate"])
            records.append(
                {
                    "page": page_name,
                    "label": box["label"],
                    "cls_id": box["cls_id"],
                    "score": float(box["score"]),
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2,
                    "width": x2 - x1,
                    "height": y2 - y1,
                }
            )
            n_boxes += 1

    print(f"  {page_name}: {n_boxes} box(es) -> {page_dir}")

print(f"\nDone. {len(records)} box(es) in total, results under {OUTPUT_DIR}")

In [ ]:
# ============================================================
# 8. Width statistics
# ============================================================
import os
import pandas as pd

df = pd.DataFrame(records)

if df.empty:
    print("No boxes were detected -- try lowering THRESHOLD in the settings cell.")
else:
    # --- per label ---------------------------------------------------
    per_label = (
        df.groupby("label")["width"]
        .agg(count="count", min_width="min", max_width="max", mean_width="mean")
        .sort_values("count", ascending=False)
        .round(2)
    )

    # --- everything together, label ignored --------------------------
    overall = pd.DataFrame(
        [
            {
                "count": len(df),
                "min_width": df["width"].min(),
                "max_width": df["width"].max(),
                "mean_width": df["width"].mean(),
            }
        ],
        index=["ALL"],
    ).round(2)

    print("=== Width statistics per label ===")
    display(per_label)

    print("=== Width statistics over all boxes (label ignored) ===")
    display(overall)

    # --- save --------------------------------------------------------
    df.to_csv(os.path.join(OUTPUT_DIR, "boxes.csv"), index=False)
    per_label.to_csv(os.path.join(OUTPUT_DIR, "width_stats_per_label.csv"))
    overall.to_csv(os.path.join(OUTPUT_DIR, "width_stats_overall.csv"))

    print(f"\nSaved boxes.csv, width_stats_per_label.csv, "
          f"width_stats_overall.csv -> {OUTPUT_DIR}")